# 🤖 03 - Entraînement du Modèle Final

**Objectif:** Entraîner plusieurs modèles sur le train set, sélectionner le meilleur sur validation, puis réentraîner le pipeline final avant l'évaluation sur le test.

In [9]:
%pip install joblib numpy pandas imbalanced-learn loguru scikit-learn xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
%pip install kagglehub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import sys
sys.path.append('..')

from src.train import run_training_pipeline


## 1. Entraînement de tous les modèles

In [13]:
# Lancement du pipeline complet corrigé
summary = run_training_pipeline()

# Récupération sécurisée des métriques (soutien aux anciennes et nouvelles structures)
best = summary.get('best_model')
results = summary.get('results', {})
metrics = summary.get('metrics', {})

res_best = results.get(best, {})
# Cas ancien: results[name] = (estimator, metrics)
if isinstance(res_best, (list, tuple)):
    details = res_best[1] if len(res_best) > 1 else {}
else:
    details = res_best or {}

# Récupérer CV/VAL/Test metrics de façon tolérante
cv_f1 = details.get('cv_f1') or details.get('best_score') or details.get('cv')
val_f1 = details.get('val_f1') or details.get('val_metrics', {}).get('f1_score')
val_pr = details.get('val_pr_auc') or details.get('val_metrics', {}).get('pr_auc')

def fmt(name, value):
    return f"{name}: {value:.4f}" if isinstance(value, (int, float)) else f"{name}: {value}"

print("\n" + "="*60)
print("RÉSULTAT FINAL DU PIPELINE")
print("="*60)
print(f"Meilleur modèle: {best}")
print(fmt('CV F1', cv_f1))
print(fmt('Validation F1', val_f1))
print(fmt('Validation PR-AUC', val_pr))
print(fmt('Test F1', metrics.get('f1_score')))
print(fmt('Test PR-AUC', metrics.get('pr_auc')))

2026-05-25 13:09:30.050 | INFO     | src.train:run_training_pipeline:205 - ============================================================
2026-05-25 13:09:30.050 | INFO     | src.train:run_training_pipeline:206 - PIPELINE D'ENTRAINEMENT (leakage-free)
2026-05-25 13:09:30.051 | INFO     | src.train:run_training_pipeline:207 - ============================================================
2026-05-25 13:09:30.051 | INFO     | src.dataset_loader:ensure_dataset:42 - Dataset deja present: c:\Users\Province Settat\Desktop\MachineLearning\fraud-detection\notebooks\..\data\raw\creditcard.csv
2026-05-25 13:09:30.727 | INFO     | src.preprocessing:load_raw_data:30 - Dataset charge: (284807, 31)
2026-05-25 13:09:31.680 | INFO     | src.preprocessing:clean_data:42 - Nettoyage: 284807 -> 283726 lignes (doublons supprimes=1081, lignes avec NaN supprimees=0)
2026-05-25 13:09:31.926 | INFO     | src.feature_engineering:add_features:68 - Feature engineering: +9 features -> 40 colonnes
2026-05-25 13:09:32.02


RÉSULTAT FINAL DU PIPELINE
Meilleur modèle: random_forest
CV F1: 0.8420
Validation F1: None
Validation PR-AUC: None
Test F1: 0.8492
Test PR-AUC: 0.8181


In [16]:
# 2. Comparaison entraînement AVEC / SANS SMOTE
from config.settings import DATA_PROCESSED, MODELS_DIR, FEATURE_COLUMNS_FILE
import json
from src.evaluate import evaluate_model

# Charger splits produits par le pipeline (tolérance aux formats)
if (DATA_PROCESSED / 'X_train.csv').exists():
    X_train = pd.read_csv(DATA_PROCESSED / 'X_train.csv')
    y_train = pd.read_csv(DATA_PROCESSED / 'y_train.csv').squeeze()
    X_val = pd.read_csv(DATA_PROCESSED / 'X_val.csv')
    y_val = pd.read_csv(DATA_PROCESSED / 'y_val.csv').squeeze()
    X_test = pd.read_csv(DATA_PROCESSED / 'X_test.csv')
    y_test = pd.read_csv(DATA_PROCESSED / 'y_test.csv').squeeze()
else:
    # Fallback: train/val/test CSV contenant la colonne cible 'Class'
    train_full = pd.read_csv(DATA_PROCESSED / 'train.csv')
    val_full = pd.read_csv(DATA_PROCESSED / 'val.csv')
    test_full = pd.read_csv(DATA_PROCESSED / 'test.csv')
    X_train = train_full.drop(columns=['Class'])
    y_train = train_full['Class']
    X_val = val_full.drop(columns=['Class'])
    y_val = val_full['Class']
    X_test = test_full.drop(columns=['Class'])
    y_test = test_full['Class']

X_train_val = pd.concat([X_train, X_val], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)

rows = []

# Cas AVEC SMOTE : utiliser les estimateurs déjà entraînés dans `results`
for name, info in results.items():
    est = info['estimator']
    # Refit sur train+val pour une évaluation finale cohérente
    est.fit(X_train_val, y_train_val)
    test_m = evaluate_model(est, X_test, y_test)
    rows.append({
        'model': name,
        'smote': True,
        'cv_f1': info.get('cv_f1'),
        'val_f1': info.get('val_f1') or info.get('val_metrics', {}).get('f1_score'),
        'val_pr_auc': info.get('val_pr_auc') or info.get('val_metrics', {}).get('pr_auc'),
        'test_f1': test_m['f1_score'],
        'test_pr_auc': test_m['pr_auc'],
        'confusion_matrix': test_m.get('confusion_matrix'),
    })

# Cas SANS SMOTE : entraîner (rapide) à partir de X_train_val
from src.train import train_single_model
for name in results.keys():
    print(f"Training (no SMOTE): {name}")
    est_no_smote, metrics_no = train_single_model(name, X_train_val, y_train_val, use_smote=False)
    test_m = evaluate_model(est_no_smote, X_test, y_test)
    rows.append({
        'model': name,
        'smote': False,
        'cv_f1': metrics_no.get('cv_f1'),
        'val_f1': None,
        'val_pr_auc': None,
        'test_f1': test_m['f1_score'],
        'test_pr_auc': test_m['pr_auc'],
        'confusion_matrix': test_m.get('confusion_matrix'),
    })

comp_df = pd.DataFrame(rows)
# Présentation: un tableau pivot côté-à-côté
pivot = comp_df.pivot_table(index='model', columns='smote', values=['cv_f1','val_f1','val_pr_auc','test_f1','test_pr_auc'])
print('\nComparaison AVEC (True) / SANS (False) SMOTE')
display(pivot)

# Affichage détaillé similaire à l'exemple demandé
summary_rows = []
for _, r in comp_df[comp_df['smote'] == True].iterrows():
    no_smote = comp_df[(comp_df['model'] == r['model']) & (comp_df['smote'] == False)].iloc[0]
    summary_rows.append([
        r['model'],
        round(r['test_f1'],4),
        round(r['test_pr_auc'],4),
        round(no_smote['test_f1'],4),
        round(no_smote['test_pr_auc'],4),
        r['cv_f1'],
        r['val_pr_auc'],
        r['confusion_matrix']
    ])

summary_table = pd.DataFrame(summary_rows, columns=['model','test_f1_with_smote','test_pr_with_smote','test_f1_no_smote','test_pr_no_smote','cv_f1','val_pr_auc','confusion_matrix'])
display(summary_table)


2026-05-25 13:28:32.015 | INFO     | src.evaluate:evaluate_model:40 - === EVALUATION (test set) ===
2026-05-25 13:28:32.015 | INFO     | src.evaluate:evaluate_model:43 -   accuracy: 0.9744
2026-05-25 13:28:32.016 | INFO     | src.evaluate:evaluate_model:43 -   precision: 0.0535
2026-05-25 13:28:32.017 | INFO     | src.evaluate:evaluate_model:43 -   recall: 0.8592
2026-05-25 13:28:32.018 | INFO     | src.evaluate:evaluate_model:43 -   f1_score: 0.1007
2026-05-25 13:28:32.019 | INFO     | src.evaluate:evaluate_model:43 -   roc_auc: 0.9603
2026-05-25 13:28:32.019 | INFO     | src.evaluate:evaluate_model:43 -   pr_auc: 0.6817
2026-05-25 13:28:32.019 | INFO     | src.evaluate:evaluate_model:44 -   TN=41408 FP=1080 FN=10 TP=61
2026-05-25 13:28:54.998 | INFO     | src.evaluate:evaluate_model:40 - === EVALUATION (test set) ===
2026-05-25 13:28:54.999 | INFO     | src.evaluate:evaluate_model:43 -   accuracy: 0.9958
2026-05-25 13:28:54.999 | INFO     | src.evaluate:evaluate_model:43 -   precisio

Training (no SMOTE): logistic_regression


2026-05-25 13:29:41.884 | INFO     | src.evaluate:evaluate_model:40 - === EVALUATION (test set) ===
2026-05-25 13:29:41.884 | INFO     | src.evaluate:evaluate_model:43 -   accuracy: 0.9763
2026-05-25 13:29:41.885 | INFO     | src.evaluate:evaluate_model:43 -   precision: 0.0575
2026-05-25 13:29:41.885 | INFO     | src.evaluate:evaluate_model:43 -   recall: 0.8592
2026-05-25 13:29:41.885 | INFO     | src.evaluate:evaluate_model:43 -   f1_score: 0.1078
2026-05-25 13:29:41.885 | INFO     | src.evaluate:evaluate_model:43 -   roc_auc: 0.9633
2026-05-25 13:29:41.886 | INFO     | src.evaluate:evaluate_model:43 -   pr_auc: 0.6746
2026-05-25 13:29:41.886 | INFO     | src.evaluate:evaluate_model:44 -   TN=41488 FP=1000 FN=10 TP=61


Training (no SMOTE): decision_tree


2026-05-25 13:30:13.835 | INFO     | src.evaluate:evaluate_model:40 - === EVALUATION (test set) ===
2026-05-25 13:30:13.835 | INFO     | src.evaluate:evaluate_model:43 -   accuracy: 0.9987
2026-05-25 13:30:13.836 | INFO     | src.evaluate:evaluate_model:43 -   precision: 0.5824
2026-05-25 13:30:13.836 | INFO     | src.evaluate:evaluate_model:43 -   recall: 0.7465
2026-05-25 13:30:13.836 | INFO     | src.evaluate:evaluate_model:43 -   f1_score: 0.6543
2026-05-25 13:30:13.837 | INFO     | src.evaluate:evaluate_model:43 -   roc_auc: 0.8729
2026-05-25 13:30:13.837 | INFO     | src.evaluate:evaluate_model:43 -   pr_auc: 0.5286
2026-05-25 13:30:13.837 | INFO     | src.evaluate:evaluate_model:44 -   TN=42450 FP=38 FN=18 TP=53


...

Training (no SMOTE): xgboost


2026-05-25 13:34:18.364 | INFO     | src.evaluate:evaluate_model:40 - === EVALUATION (test set) ===
2026-05-25 13:34:18.365 | INFO     | src.evaluate:evaluate_model:43 -   accuracy: 0.9996
2026-05-25 13:34:18.365 | INFO     | src.evaluate:evaluate_model:43 -   precision: 0.9492
2026-05-25 13:34:18.366 | INFO     | src.evaluate:evaluate_model:43 -   recall: 0.7887
2026-05-25 13:34:18.366 | INFO     | src.evaluate:evaluate_model:43 -   f1_score: 0.8615
2026-05-25 13:34:18.366 | INFO     | src.evaluate:evaluate_model:43 -   roc_auc: 0.9794
2026-05-25 13:34:18.367 | INFO     | src.evaluate:evaluate_model:43 -   pr_auc: 0.8201
2026-05-25 13:34:18.367 | INFO     | src.evaluate:evaluate_model:44 -   TN=42485 FP=3 FN=15 TP=56



Comparaison AVEC (True) / SANS (False) SMOTE


cv_f1             test_f1           test_pr_auc  \
smote                   False     True      False     True        False   
model                                                                     
decision_tree        0.641345  0.470599  0.654321  0.373239    0.528646   
logistic_regression  0.117322  0.116883  0.107774  0.100660    0.674614   
random_forest        0.853151  0.842033  0.806452  0.827068    0.819591   
xgboost              0.851567  0.695769  0.861538  0.595960    0.820088   

                               
smote                   True   
model                          
decision_tree        0.324148  
logistic_regression  0.681737  
random_forest        0.826526  
xgboost              0.790505

,model,test_f1_with_smote,test_pr_with_smote,test_f1_no_smote,test_pr_no_smote,cv_f1,val_pr_auc,confusion_matrix
0,logistic_regression,0.1007,0.6817,0.1078,0.6746,0.116883,None,"[[41408, 1080], [10, 61]]"
1,decision_tree,0.3732,0.3241,0.6543,0.5286,0.470599,None,"[[42328, 160], [18, 53]]"
2,random_forest,0.8271,0.8265,0.8065,0.8196,0.842033,None,"[[42481, 7], [16, 55]]"
3,xgboost,0.5960,0.7905,0.8615,0.8201,0.695769,None,"[[42420, 68], [12, 59]]"


## 3. Feature Importance (XGBoost)

In [19]:
# Feature importance pour XGBoost (pipeline sauvegardé)
import joblib
import pandas as pd
from config.settings import MODELS_DIR, FEATURE_COLUMNS_FILE
import json

feat_file = MODELS_DIR / FEATURE_COLUMNS_FILE
if feat_file.exists():
    feat_cols = json.load(open(feat_file))
else:
    feat_cols = None

xgb_path = MODELS_DIR / 'xgboost.joblib'
if not xgb_path.exists():
    xgb_path = MODELS_DIR / 'best_model.joblib'

if xgb_path.exists():
    xgb_pipe = joblib.load(xgb_path)
    clf = xgb_pipe.named_steps.get('classifier')
    if hasattr(clf, 'feature_importances_'):
        imp = pd.Series(clf.feature_importances_, index=feat_cols if feat_cols else X_train.columns)
    else:
        # fallback: booster importance mapping f0.. -> values
        try:
            booster = clf.get_booster()
            score = booster.get_score(importance_type='gain')
            imp_list = [score.get(f'f{i}', 0) for i in range(len(feat_cols or X_train.columns))]
            imp = pd.Series(imp_list, index=feat_cols if feat_cols else X_train.columns)
        except Exception as e:
            print("Impossible d'extraire les importances:", e)
            imp = pd.Series([], dtype=float)
    imp = imp.sort_values(ascending=False)
    print('\nTop 20 features (XGBoost)')
    display(imp.head(20))
else:
    print("Aucun modèle XGBoost trouvé dans models/, exécutez le pipeline d'entraînement.")



Top 20 features (XGBoost)


risk_score      0.092025
V18             0.079118
V1              0.070753
v_std           0.068430
V4              0.064826
V8              0.063025
V14             0.039844
amount_log      0.034447
V26             0.033581
V2              0.031258
V21             0.030264
day_period      0.029968
V9              0.029473
V25             0.025781
V17             0.025300
V22             0.024848
V11             0.023079
V16             0.017706
V3              0.017594
amount_x_v14    0.015224
dtype: float32

## 4. Modèles sauvegardés

Les modèles sont sauvegardés dans `models/`:
- `best_model.joblib` → Pipeline final réentraîné sur train + validation
- `logistic_regression.joblib`
- `decision_tree.joblib`
- `random_forest.joblib`
- `xgboost.joblib`
- `scaler.joblib` → Scaler du pipeline final